# Xenium sc-SVC Reconstruction Impact

A route-specific, reproducible description of reconstruction-associated partition and local-composition changes.

## 1. Route semantics and carrier audit

Raw Leiden is compared with final SVC clusters; the spatial carrier expression-identity audit prevents an artificial expression edge. **Evidence boundary:** these are paired representation and spatial-pattern observations; they do not establish a mechanism, biological truth, or clinical meaning.

In [ ]:
import os
import sys
import warnings
from pathlib import Path

os.environ.setdefault("KMP_WARNINGS", "0")
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("KMP_USE_SHM", "0")
os.environ.setdefault("KMP_AFFINITY", "disabled")
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message="If reg_type = entropy, then the matrix c is overwritten")
warnings.filterwarnings("ignore", message="Changing the sparsity structure of a csr_matrix is expensive.*")

PROJECT_ROOT = Path(os.environ.get("REVISE_REPOSITORY_ROOT", Path.cwd())).resolve()
if not (PROJECT_ROOT / "configs").is_dir():
    PROJECT_ROOT = Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import anndata as ad
import matplotlib.pyplot as plt
from matplotlib.colors import BoundaryNorm, ListedColormap, TwoSlopeNorm
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display

from revise.analysis.reconstruction_impact import (
    compute_anatomy_regions,
    compute_spatial_impact,
    file_sha256,
    load_reconstruction_impact_config,
    map_raw_level2_labels,
    run_partition_analysis,
    select_raw_level1_parent_cohort,
    write_analysis_artifacts,
    write_anatomy_artifacts,
    write_partition_artifacts,
    write_raw_level2_artifacts,
    write_spatial_artifacts,
)

ROOT = PROJECT_ROOT

ANATOMY_ORDER = ["Tumor", "Normal", "Interface", "Other"]
ANATOMY_COLORS = {"Tumor": "#d73027", "Normal": "#4575b4", "Interface": "#984ea3", "Other": "#d9d9d9"}

def coordinates(adata_obj):
    return pd.DataFrame(adata_obj.obsm["spatial"], index=adata_obj.obs_names, columns=["x", "y"])

def compact_iqr(values):
    values = pd.Series(values).dropna()
    if values.empty:
        return "NA"
    return f"{values.median():.3g} [{values.quantile(.25):.3g}, {values.quantile(.75):.3g}]"

def save_figure(fig, name):
    figure_dir = OUTPUT_DIR / "figures"
    figure_dir.mkdir(parents=True, exist_ok=True)
    fig.savefig(figure_dir / f"{name}.png", dpi=180, bbox_inches="tight")

def set_spatial_axes(ax):
    ax.set(xlim=TISSUE_XLIM, ylim=TISSUE_YLIM, xlabel="x (µm)", ylabel="y (µm)")
    ax.set_aspect("equal", adjustable="box")

def window_field(ax, frame, value, *, side_um, cmap="viridis", norm=None, title="", categorical=False):
    valid = frame.dropna(subset=[value])
    if valid.empty:
        set_spatial_axes(ax); ax.set_title(title + " (no valid windows)"); return None
    x_indices = np.arange(valid["window_x_index"].min(), valid["window_x_index"].max() + 1)
    y_indices = np.arange(valid["window_y_index"].min(), valid["window_y_index"].max() + 1)
    grid = np.full((len(y_indices), len(x_indices)), np.nan)
    for row in valid.itertuples():
        grid[int(row.window_y_index - y_indices[0]), int(row.window_x_index - x_indices[0])] = getattr(row, value)
    x_edges = ORIGIN_UM[0] + np.arange(x_indices[0], x_indices[-1] + 2) * side_um
    y_edges = ORIGIN_UM[1] + np.arange(y_indices[0], y_indices[-1] + 2) * side_um
    image = ax.pcolormesh(x_edges, y_edges, grid, shading="flat", cmap=cmap, norm=norm)
    set_spatial_axes(ax); ax.set_title(title)
    return image

def anatomy_map(ax, focus=None):
    regions = ANATOMY.anatomy_windows.copy()
    codes = regions["level1_region"].map({name: i for i, name in enumerate(ANATOMY_ORDER)})
    if focus is not None:
        codes = np.where(regions["level1_region"].eq(focus), ANATOMY_ORDER.index(focus), ANATOMY_ORDER.index("Other"))
    frame = regions.assign(anatomy_code=codes)
    cmap = ListedColormap([ANATOMY_COLORS[name] for name in ANATOMY_ORDER])
    image = window_field(ax, frame, "anatomy_code", side_um=float(ANATOMY.scale_audit["main_window_side_um"]), cmap=cmap, norm=BoundaryNorm(np.arange(-.5, 4.5), 4), title=focus or "Tumor / Normal / Interface / Other")
    if image is not None and focus is None:
        colorbar = plt.colorbar(image, ax=ax, ticks=range(4), fraction=.046, pad=.04)
        colorbar.ax.set_yticklabels(ANATOMY_ORDER)

def plot_support_curve(impact, title):
    table = impact.support_sensitivity
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))
    cells = table["window_side_length"] / CELL_EQUIVALENT_UM
    axes[0].plot(cells, table["retained_parent_unit_fraction"], marker="o")
    axes[0].axvline(impact.scale_audit["main_window_cells_per_side"], color="black", ls="--")
    axes[0].set(xlabel="cells per side", ylabel="retained-unit fraction", title="retention")
    axes[1].plot(cells, table["valid_window_fraction"], marker="o")
    axes[1].axvline(impact.scale_audit["main_window_cells_per_side"], color="black", ls="--")
    axes[1].set(xlabel="cells per side", ylabel="valid-window fraction", title="valid windows")
    fig.suptitle(title); fig.tight_layout(); return fig

def plot_contingency(comparisons, title):
    fig, axes = plt.subplots(1, len(comparisons), figsize=(4.5 * len(comparisons), 3.8), squeeze=False)
    for ax, (scope, comparison) in zip(axes[0], comparisons.items()):
        table = comparison.contingency.div(comparison.contingency.sum(axis=1), axis=0).fillna(0)
        sns.heatmap(table, ax=ax, cmap="mako", vmin=0, vmax=1, cbar=True)
        ax.set(title=scope, xlabel="Reconstructed / Final", ylabel="Raw")
    fig.suptitle(title); fig.tight_layout(); return fig

def plot_changed_units(runs, title):
    fig, axes = plt.subplots(1, len(runs), figsize=(4.5 * len(runs), 4), squeeze=False)
    for ax, (scope, run) in zip(axes[0], runs.items()):
        frame = run["impact"].unit_assignments
        colors = np.where(frame["unit_changed"], "#d73027", "#d9d9d9")
        ax.scatter(frame["x"], frame["y"], c=colors, s=1, linewidths=0, rasterized=True)
        set_spatial_axes(ax); ax.set_title(scope)
    fig.suptitle(title); fig.tight_layout(); return fig

def plot_cluster_pair(run, title):
    frame = run["impact"].unit_assignments
    mapping = run["comparison"].mapping.set_index("recon_cluster")["raw_cluster"].to_dict()
    raw_values = sorted(frame["raw_cluster"].unique())
    palette = {value: color for value, color in zip(raw_values, sns.color_palette("tab20", len(raw_values)))}
    recon_colors = [palette.get(mapping.get(value), "#111111") for value in frame["reconstructed_cluster"]]
    fig, axes = plt.subplots(1, 2, figsize=(9, 4))
    axes[0].scatter(frame["x"], frame["y"], c=frame["raw_cluster"].map(palette), s=1, linewidths=0, rasterized=True)
    axes[1].scatter(frame["x"], frame["y"], c=recon_colors, s=1, linewidths=0, rasterized=True)
    for ax, label in zip(axes, ["Raw Leiden", "Reconstructed / Final"]):
        set_spatial_axes(ax); ax.set_title(label)
    fig.suptitle(title + " — matched clusters share colors"); fig.tight_layout(); return fig

def plot_reconstructed_clusters(run, title):
    frame = run["impact"].unit_assignments
    values = sorted(frame["reconstructed_cluster"].unique())
    palette = {value: color for value, color in zip(values, sns.color_palette("tab20", len(values)))}
    fig, ax = plt.subplots(figsize=(5.2, 4.4))
    ax.scatter(frame["x"], frame["y"], c=frame["reconstructed_cluster"].map(palette), s=1, linewidths=0, rasterized=True)
    set_spatial_axes(ax); ax.set_title(title); fig.tight_layout(); return fig

def plot_metric_comparison(run, metric, title):
    metrics = run["impact"].window_metrics.loc[lambda x: x["valid_window"]].copy()
    side = float(run["impact"].scale_audit["main_window_side_um"])
    state_columns = [f"{metric}_raw", f"{metric}_recon", f"{metric}_level2"]
    state_max = max(float(metrics[state_columns].max().max()), 1.0)
    state_min = 0.0 if metric == "evenness" else 1.0
    state_max = 1.0 if metric == "evenness" else state_max
    delta_columns = [f"delta_{metric}_vs_raw_leiden", f"delta_{metric}_vs_raw_level2"]
    delta_max = max(float(metrics[delta_columns].abs().max().max()), 0.01)
    state_norm = plt.Normalize(state_min, state_max)
    delta_norm = TwoSlopeNorm(vcenter=0, vmin=-delta_max, vmax=delta_max)
    panels = [
        (f"{metric}_raw", "Raw Leiden", "viridis", state_norm),
        (f"{metric}_recon", "Reconstructed", "viridis", state_norm),
        (f"delta_{metric}_vs_raw_leiden", "Recon − Raw Leiden", "coolwarm", delta_norm),
        (f"{metric}_level2", "Raw Level2", "viridis", state_norm),
        (f"{metric}_recon", "Reconstructed", "viridis", state_norm),
        (f"delta_{metric}_vs_raw_level2", "Recon − Raw Level2", "coolwarm", delta_norm),
    ]
    fig, axes = plt.subplots(2, 3, figsize=(13.5, 8.2), sharex=True, sharey=True)
    for ax, (column, label, cmap, norm) in zip(axes.flat, panels):
        image = window_field(ax, metrics, column, side_um=side, cmap=cmap, norm=norm, title=label)
        if image is not None:
            plt.colorbar(image, ax=ax, fraction=.046, pad=.04)
    fig.suptitle(title); fig.tight_layout(); return fig

def plot_high_diversity(run, title):
    metrics = run["impact"].window_metrics.copy()
    side = float(run["impact"].scale_audit["main_window_side_um"])
    valid = metrics.loc[metrics["valid_window"]].copy()
    maximum = max(float(valid["neff_recon"].max()), 1.0)
    fig, axes = plt.subplots(1, 2, figsize=(9.2, 4.2), sharex=True, sharey=True)
    image = window_field(axes[0], valid, "neff_recon", side_um=side, cmap="viridis", norm=plt.Normalize(1, maximum), title="Reconstructed Neff")
    if image is not None:
        plt.colorbar(image, ax=axes[0], fraction=.046, pad=.04)
    threshold_ok = run["impact"].state_threshold.get("status") == "ok"
    mask = valid.assign(region_mask=valid["in_state_region"].fillna(False).astype(float))
    if threshold_ok:
        image = window_field(axes[1], mask, "region_mask", side_um=side, cmap=ListedColormap(["#efefef", "#fdae61"]), norm=BoundaryNorm([-0.5, 0.5, 1.5], 2), title="High-diversity Region")
        if image is not None:
            colorbar = plt.colorbar(image, ax=axes[1], ticks=[0, 1], fraction=.046, pad=.04)
            colorbar.ax.set_yticklabels(["outside", "inside"])
    else:
        window_field(axes[1], mask, "region_mask", side_um=side, cmap=ListedColormap(["#efefef"]), norm=BoundaryNorm([-0.5, 0.5], 1), title="No stable Region threshold")
    fig.suptitle(title); fig.tight_layout(); return fig

def raw_level2_mapping(raw_parent, parent):
    mapping_config = CONFIG["raw_level2_mapping"]
    level1 = mapping_config["level1_column"]
    reference_context = ad.read_h5ad(ROOT / mapping_config["reference_h5ad"], backed="r")
    reference_mask = reference_context.obs[level1].astype(str).str.replace("/", "_", regex=False).eq(parent)
    reference_filter = mapping_config.get("reference_filter")
    if reference_filter:
        reference_mask &= reference_context.obs[reference_filter["column"]].astype(str).eq(str(reference_filter["value"]))
    reference_parent = reference_context[reference_mask].to_memory()
    reference_context.file.close()
    pot = mapping_config.get("pot", {}); tacco = mapping_config.get("tacco", {})
    return map_raw_level2_labels(raw_parent, reference_parent, parent_value=parent, method=mapping_config["method"], level1_col=level1, level2_col=mapping_config["level2_column"], reference_filter_column=(reference_filter or {}).get("column"), reference_filter_value=(reference_filter or {}).get("value"), pot_reg=pot.get("reg", .1), pot_reg_m=pot.get("reg_m", 0.), pot_reg_type=pot.get("reg_type", "entropy"), tacco_multi_center=tacco.get("multi_center"), tacco_lamb=tacco.get("lamb"))

def extent_table(impact, column):
    source = impact.region_extent_by_anatomy if column == "in_state_region" else impact.gain_region_extent_by_anatomy
    return source.loc[source["level1_region"].isin(["Overall", "Tumor", "Normal", "Interface"])]


In [ ]:
CACHE_PATH = os.environ.get("RECONSTRUCTION_IMPACT_CACHE_PATH")
if CACHE_PATH:
    import pickle
    with Path(CACHE_PATH).open("rb") as handle:
        globals().update(pickle.load(handle))
else:
    CONFIG = load_reconstruction_impact_config(ROOT / "configs" / "analysis" / "reconstruction_impact_xenium_p2crc_fibroblast.yaml")
    OUTPUT_DIR = Path(os.environ.get("REVISE_ANALYSIS_OUTPUT_ROOT", CONFIG["output"]["dir"]))
    CELL_EQUIVALENT_UM = 8.0
    LEVEL1 = CONFIG["context"]["level1_column"]
    raw_path = ROOT / CONFIG["context"]["h5ad"]
    raw_context = ad.read_h5ad(raw_path, backed="r")
    full_coordinates = coordinates(raw_context); full_level1 = raw_context.obs[LEVEL1].astype(str).copy()
    TISSUE_COORDINATES_UM = full_coordinates * CONFIG["spatial_region"]["microns_per_coordinate"]
    TISSUE_XLIM = tuple(TISSUE_COORDINATES_UM["x"].agg(["min", "max"])); TISSUE_YLIM = tuple(TISSUE_COORDINATES_UM["y"].agg(["min", "max"])); ORIGIN_UM = (TISSUE_XLIM[0], TISSUE_YLIM[0])
    ANATOMY = compute_anatomy_regions(full_coordinates=full_coordinates, full_level1_labels=full_level1, microns_per_coordinate=CONFIG["spatial_region"]["microns_per_coordinate"], candidate_window_sides_um=CONFIG["spatial_region"]["candidate_window_sides_um"], min_parent_units=CONFIG["spatial_region"]["min_parent_units"], cell_equivalent_um=CELL_EQUIVALENT_UM, **CONFIG["spatial_region"]["anatomy_region"])
    write_anatomy_artifacts(OUTPUT_DIR, ANATOMY)
    RUNS = {}
    for comparison_config in CONFIG["partition_change"]["comparisons"]:
        parent = comparison_config["name"]
        recon_parent = ad.read_h5ad(ROOT / comparison_config["reconstructed_spatial_h5ad"])
        parent_ids, cohort_audit = select_raw_level1_parent_cohort(raw_context.obs[LEVEL1], recon_parent.obs_names, parent_value=parent)
        recon_parent = recon_parent[parent_ids].copy()
        raw_parent = raw_context[parent_ids].to_memory()
        partition = run_partition_analysis(raw_parent, recon_parent, level1_col=LEVEL1, final_cluster_key=comparison_config["reconstructed_cluster_key"], route_kind="sc_svc", resolution_mode="fixed_within_level1", within_level1_resolution=CONFIG["partition_change"]["within_level1_resolution"], random_state=CONFIG["partition_change"]["random_state"], n_top_genes=CONFIG["partition_change"]["n_top_genes"])
        comparison = next(iter(partition.comparisons.values())); assignments = comparison.assignments.set_index("unit_id")
        level2_mapping = raw_level2_mapping(raw_parent, parent)
        impact = compute_spatial_impact(full_coordinates=full_coordinates, full_level1_labels=full_level1, paired_coordinates=coordinates(raw_parent), raw_labels=assignments["raw_cluster"], raw_level2_labels=level2_mapping.labels, reconstructed_labels=assignments["recon_cluster"], unit_changed=assignments["unit_changed"], microns_per_coordinate=CONFIG["spatial_region"]["microns_per_coordinate"], candidate_window_sides_um=CONFIG["spatial_region"]["candidate_window_sides_um"], min_parent_units=CONFIG["spatial_region"]["min_parent_units"], rarefaction_draws=CONFIG["spatial_region"]["rarefaction_draws"], threshold_bootstraps=CONFIG["spatial_region"]["threshold_bootstraps"], cell_equivalent_um=CELL_EQUIVALENT_UM, anatomy_analysis=ANATOMY, **CONFIG["spatial_region"]["anatomy_region"])
        RUNS[parent] = {"partition": partition, "comparison": comparison, "impact": impact, "raw_level2": level2_mapping, "cohort_audit": cohort_audit, "available": cohort_audit["carrier_units"], "used": raw_parent.n_obs, "expression_h5ad": comparison_config["expression_h5ad"]}
    for name, run in RUNS.items(): write_partition_artifacts(OUTPUT_DIR / name, run["partition"]); write_raw_level2_artifacts(OUTPUT_DIR / name, run["raw_level2"]); write_spatial_artifacts(OUTPUT_DIR / name, run["impact"])
    manifest = {"analysis_contract_version": 4, "route": "sc_svc", "anatomy_scale": ANATOMY.scale_audit, "inputs": {"raw": {"path": CONFIG["context"]["h5ad"], "sha256": file_sha256(raw_path)}, "level2_reference": {"path": CONFIG["raw_level2_mapping"]["reference_h5ad"], "sha256": file_sha256(ROOT / CONFIG["raw_level2_mapping"]["reference_h5ad"])}}, "parents": {name: {"expression_h5ad": run["expression_h5ad"], "spatial_expression_identical": run["partition"].representation_audit.get("spatial_expression_identical"), "raw_level2_mapping": run["raw_level2"].audit, "cohort_audit": run["cohort_audit"]} for name, run in RUNS.items()}}
    input_audit = pd.DataFrame([{"role": "Raw full Level1 context", "units": raw_context.n_obs, "paired": True}, *[{"role": f"{name} spatial carrier after Raw Level1 audit", "units": run["used"], "excluded_units": run["cohort_audit"]["excluded_raw_level1_mismatch"], "paired": True} for name, run in RUNS.items()], {"role": "Raw Level2 single-cell reference", "units": sum(run["raw_level2"].audit["n_reference_cells"] for run in RUNS.values()), "paired": False}])
    write_analysis_artifacts(OUTPUT_DIR, config=CONFIG, manifest=manifest, input_audit=input_audit)
    raw_context.file.close()

In [ ]:
display(input_audit)
display(pd.DataFrame([{"8 µm cell-equivalent": CELL_EQUIVALENT_UM, "coordinate to µm": CONFIG["spatial_region"]["microns_per_coordinate"], "window candidates (µm)": str(CONFIG["spatial_region"]["candidate_window_sides_um"])}]))

### Partition preprocessing contract

For sp-SVC, observation and gene QC are defined on Raw counts and applied to both paired carriers. Raw-derived Seurat-v3 HVGs are then shared by Raw and reconstructed expression. Leiden uses the explicitly recorded igraph backend and seed. This prevents reconstructed expression from defining the Raw feature space.

In [ ]:
if CONFIG["route_kind"] == "sp_svc":
    audit = GLOBAL_PARTITION.audit
    display(pd.DataFrame([{"sampled units": audit["input_units"], "Raw-QC retained units": audit["n_units"], "excluded units": audit["excluded_raw_qc_units"], "retained shared genes": audit["n_shared_genes"], "features": audit["n_features"], "feature selection": audit["feature_selection"], "Leiden backend": audit["leiden_backend"], "seed": audit["leiden_random_state"]}]))
else:
    first = next(iter(RUNS.values()))["partition"]
    display(pd.DataFrame([{"carrier expression identity": all(run["partition"].representation_audit.get("spatial_expression_identical", False) for run in RUNS.values()), "feature selection": first.audit["feature_selection"], "Leiden backend": first.audit["leiden_backend"], "seed": first.audit["leiden_random_state"]}]))

### Local diversity metric definitions

For every valid parent window and every paired rarefaction draw: **Kobs** counts observed subtype/cluster labels; **Neff** is the abundance-aware effective cluster count; **evenness = Neff / Kobs** measures balance conditional on richness. The same sampled units are used for Raw Leiden, Raw Level2 and reconstructed assignments. These definitions apply identically to all three parent sections.

## 2. Partition complexity diagnostic

This diagnostic retains the Raw resolution. It asks whether reconstruction yields a finer or reorganized partition; it is not the controlled assignment-change headline.

In [ ]:
complexity_rows = []
for name, run in RUNS.items():
    row = next(iter(run["partition"].complexity_comparisons.values())).summary.iloc[0]
    complexity_rows.append({"scope": name, "Raw K": row.n_raw_clusters, "Reconstructed K at Raw resolution": row.n_recon_clusters, "ARI": row["ARI"]})
if 'GLOBAL_PARTITION' in globals():
    row = next(iter(GLOBAL_PARTITION.complexity_comparisons.values())).summary.iloc[0]
    complexity_rows.insert(0, {"scope": "Global", "Raw K": row.n_raw_clusters, "Reconstructed K at Raw resolution": row.n_recon_clusters, "ARI": row["ARI"]})
complexity_table = pd.DataFrame(complexity_rows); display(complexity_table.round(3))
display(Markdown("A larger reconstructed K supports the focused structural observation: coarse expression structure can remain, while smaller or boundary-ambiguous Raw clusters are split or reorganized."))

## 3. Matched-complexity change and Level1 localization

For each comparison, the reconstructed resolution is independently selected to match Raw K as closely as possible. Hungarian matching is then global and is reused for every Level1 proportion and spatial map.

In [ ]:
COMP = ({name: run["comparison"] for name, run in RUNS.items()})
COMP_STATUS = {name: run["partition"].matched_cluster_status for name, run in RUNS.items()}
if 'GLOBAL_PARTITION' in globals():
    COMP = {"Global": next(iter(GLOBAL_PARTITION.comparisons.values())), **COMP}
    COMP_STATUS = {"Global": GLOBAL_PARTITION.matched_cluster_status, **COMP_STATUS}
plot_comp = {(name if COMP_STATUS[name] != "unmatched_cluster_complexity" else f"{name} [audit: unmatched K]"): comparison for name, comparison in COMP.items()}
fig = plot_contingency(plot_comp, "Cluster-count-controlled contingency (unmatched K is audit only)")
save_figure(fig, "matched_k_contingency"); plt.show()

In [ ]:
change_rows=[]
for name, comparison in COMP.items():
    row=comparison.summary.iloc[0]; status=COMP_STATUS[name]; change_rows.append({"scope": name, "paired units": row.n_units, "Raw K": row.n_raw_clusters, "Recon K": row.n_recon_clusters, "status": status, "headline eligible": status != "unmatched_cluster_complexity", "ST-unit change": row.st_unit_change_fraction, "balanced change": row.balanced_cluster_change, "ARI": row["ARI"]})
change_table=pd.DataFrame(change_rows); display(change_table.round(4))
if (change_table["headline eligible"] == False).any(): display(Markdown("Rows marked `unmatched_cluster_complexity` are retained as audits; their change metrics are not interpreted as matched-K headlines."))

In [ ]:
level1_tables=[]
for name, run in RUNS.items():
    table=run["partition"].change_by_level1.copy(); table.insert(0, "scope", name); level1_tables.append(table)
if 'GLOBAL_PARTITION' in globals():
    table=GLOBAL_PARTITION.change_by_level1.copy(); table.insert(0, "scope", "Global"); level1_tables.insert(0, table)
level1_change=pd.concat(level1_tables, ignore_index=True)
fig, ax=plt.subplots(figsize=(7, 5.8))
if CONFIG["route_kind"] == "sc_svc":
    parent_order=["Fibroblast", "Mono_Macro", "T"]
    plot_table=(level1_change.loc[level1_change["scope"].isin(parent_order)].sort_values("scope", key=lambda x: x.map({name:i for i,name in enumerate(parent_order)})).drop_duplicates("scope"))
    lower=plot_table["change_fraction"]-plot_table["wilson_ci_lower"]; upper=plot_table["wilson_ci_upper"]-plot_table["change_fraction"]
    ax.bar(plot_table["scope"], plot_table["change_fraction"], color="#4c78a8", yerr=np.vstack([lower, upper]), capsize=4)
    ax.set(title="Parent-internal matched-K reassignment", ylabel="reassigned fraction")
    display_table=plot_table.loc[:, ["scope","level1","total_units","changed_units","change_fraction","wilson_ci_lower","wilson_ci_upper"]]
else:
    plot_table=level1_change.query("scope == 'Global'")
    if GLOBAL_PARTITION.matched_cluster_status == "unmatched_cluster_complexity":
        row=next(iter(GLOBAL_PARTITION.comparisons.values())).summary.iloc[0]
        ax.axis("off"); ax.text(.5,.55,f"No headline Level1 change estimate\nRaw K={int(row.n_raw_clusters)}, closest Recon K={int(row.n_recon_clusters)}",ha="center",va="center",fontsize=13)
        display_table=pd.DataFrame([{"scope":"Global","status":GLOBAL_PARTITION.matched_cluster_status,"Raw K":int(row.n_raw_clusters),"closest Recon K":int(row.n_recon_clusters),"Level1 audit artifact":"global/st_unit/change_by_level1.csv"}])
    else:
        plot_table=plot_table.loc[plot_table["level1"] != "Overall"].sort_values("change_fraction")
        lower=plot_table["change_fraction"]-plot_table["wilson_ci_lower"]; upper=plot_table["wilson_ci_upper"]-plot_table["change_fraction"]
        ax.barh(plot_table["level1"], plot_table["change_fraction"], color="#4c78a8", xerr=np.vstack([lower, upper]), capsize=3)
        ax.set(title="Level1 matched-K change fraction", xlabel="changed fraction", ylabel="")
        focus_level1=["Overall","Fibroblast","Mono/Macro","T"]
        focus=level1_change.loc[(level1_change["scope"] == "Global") & level1_change["level1"].isin(focus_level1)]
        top=plot_table.nlargest(3, "change_fraction")
        display_table=pd.concat([focus,top],ignore_index=True).drop_duplicates("level1").loc[:, ["scope","level1","total_units","changed_units","change_fraction","wilson_ci_lower","wilson_ci_upper"]]
save_figure(fig, "level1_change_fraction"); plt.show()
display(display_table.round(4))

In [ ]:
fig=plot_changed_units(RUNS, "Changed-unit spatial localization")
save_figure(fig, "changed_units"); plt.show()
key_mapping=[]
for name, run in RUNS.items():
    table=run["comparison"].mapping.copy(); table.insert(0, "scope", name); key_mapping.append(table.nlargest(2, "overlap_n"))
display(pd.concat(key_mapping, ignore_index=True).head(6))

## 4. Level1 anatomy Regions

This is an independent, full-tissue macro context. It is neither sampled with the VisiumHD partition switch nor used to tune cluster complexity, parent windows, or diversity thresholds.

In [ ]:
fig=plot_support_curve(ANATOMY, "Route-level anatomy regions: occupancy decision")
save_figure(fig, "anatomy_window_decision"); plt.show()
display(pd.DataFrame([{"selected cells per side": ANATOMY.scale_audit["main_window_cells_per_side"], "selected side (µm)": ANATOMY.scale_audit["main_window_side_um"], "valid windows": int(ANATOMY.anatomy_windows.shape[0]), "retained units": ANATOMY.support_selection.get("retained_parent_unit_fraction")}]))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 9))
anatomy_map(axes[0,0]); anatomy_map(axes[0,1], "Tumor"); anatomy_map(axes[1,0], "Normal"); anatomy_map(axes[1,1], "Interface")
fig.suptitle("Level1 anatomy Regions"); fig.tight_layout()
save_figure(fig, "anatomy_regions"); plt.show()
display(ANATOMY.anatomy_context_summary.loc[ANATOMY.anatomy_context_summary["level1_region"].isin(["Tumor","Normal","Interface"]), ["level1_region","full_level1_units","tissue_windows","area_um2","area_fraction"]])

## 5. Fibroblast internal diversity and Regions

This parent is analysed after the fixed macro anatomy. Raw Level2 is mapped from the original Raw expression after Level1 parent selection; reconstructed-carrier Level2 labels are not used as this baseline.

### Cohort and window decision

The cell-equivalent side is fixed at 8 µm. The selected side comes only from parent occupancy; support is four units, and each of 200 paired draws samples four units.

In [ ]:
RUN = RUNS["Fibroblast"]; IMPACT = RUN["impact"]
fig = plot_support_curve(IMPACT, "Fibroblast: occupancy-only window decision")
save_figure(fig, "fibroblast_window_decision"); plt.show()
occupancy = IMPACT.unit_assignments.groupby("window_id").size()
sampled = RUN.get("sampled", RUN["used"])
decision = pd.DataFrame([{"available units": RUN["available"], "sampled units": sampled, "Raw-QC retained units": RUN["used"], "sampling mode": "full" if RUN["available"] == sampled else "deterministic 30k", "seed": CONFIG["partition_change"]["random_state"], "window side (cells per side)": IMPACT.scale_audit["main_window_cells_per_side"], "window side (µm)": IMPACT.scale_audit["main_window_side_um"], "occupancy median [Q1,Q3]": compact_iqr(occupancy), "minimum support": IMPACT.scale_audit["min_parent_units"], "rarefaction units": IMPACT.scale_audit["min_parent_units"], "paired draws": IMPACT.scale_audit["rarefaction_draws"]}])
display(decision)

### Internal-state baselines and matched-K assignments

The uniform Raw Level1 label is only the unexpanded baseline (`Kobs = Neff = evenness = 1`). The two informative Raw baselines are (i) expression-derived matched-K Leiden and (ii) reference-derived Raw Level2. Both are compared with the same reconstructed assignment on identical windows and identical rarefaction draws.

In [ ]:
METRICS = RUN["impact"].window_metrics.copy(); METRICS.attrs["side_um"] = IMPACT.scale_audit["main_window_side_um"]
fig = plot_reconstructed_clusters(RUN, "Fibroblast: reconstructed internal state")
save_figure(fig, "fibroblast_reconstructed_clusters"); plt.show()
display(pd.DataFrame([{"baseline": "Raw Level1 (uniform parent)", "Kobs": 1.0, "Neff": 1.0, "evenness": 1.0}, {"baseline": "Raw Level2 mapping", "Kobs": RUN["raw_level2"].audit["n_mapped_level2"], "Neff": "window-specific", "evenness": "window-specific"}]))
display(pd.DataFrame([RUN["raw_level2"].audit]).loc[:, ["source", "method", "n_raw_units", "n_reference_cells", "n_reference_level2", "n_mapped_level2", "n_missing"]])

### Matched-K reconstruction-associated internal structure

This is a **parent-internal reassignment** analysis, not a Level1 identity change. Raw Leiden is tuned to approximately the same cluster count as the reconstructed/final assignment. Hungarian matching controls label permutation for unit-change metrics; the diversity matrices below use the original cluster compositions within each window.

In [ ]:
fig = plot_cluster_pair(RUN, "Fibroblast: matched-K cluster assignment")
save_figure(fig, "fibroblast_matched_clusters"); plt.show()
row = RUN["comparison"].summary.iloc[0]
display(pd.DataFrame([{"paired units": row.n_units, "Raw K": row.n_raw_clusters, "Recon K": row.n_recon_clusters, "parent-internal reassignment": row.st_unit_change_fraction, "balanced change": row.balanced_cluster_change, "ARI": row["ARI"]}]).round(4))

### Kobs: local subtype richness

`Kobs` is the number of cluster labels observed in a rarefied window. Here it asks how many distinct internal states coexist locally, without accounting for whether one state dominates.

In [ ]:
fig = plot_metric_comparison(RUN, "k_obs", "Fibroblast: Kobs evidence matrix")
save_figure(fig, "fibroblast_kobs_matrix"); plt.show()
valid = METRICS.loc[METRICS.valid_window]
display(pd.DataFrame([{"baseline": "Raw Leiden", "state median [Q1,Q3]": compact_iqr(valid["k_obs_raw"]), "Recon minus baseline": compact_iqr(valid["delta_k_obs_vs_raw_leiden"])}, {"baseline": "Raw Level2", "state median [Q1,Q3]": compact_iqr(valid["k_obs_level2"]), "Recon minus baseline": compact_iqr(valid["delta_k_obs_vs_raw_level2"])}]))
display(Markdown(f"Across valid windows, reconstructed Kobs is **{compact_iqr(valid['k_obs_recon'])}**. Its median difference is **{valid['delta_k_obs_vs_raw_leiden'].median():.3g}** versus Raw Leiden and **{valid['delta_k_obs_vs_raw_level2'].median():.3g}** versus Raw Level2."))

### Neff: abundance-aware effective subtype count

`Neff = exp(Shannon entropy)` converts subtype composition into an effective number of equally abundant clusters. It is the primary local-diversity state because it discounts rare labels and single-cluster dominance.

In [ ]:
fig = plot_metric_comparison(RUN, "neff", "Fibroblast: Neff evidence matrix")
save_figure(fig, "fibroblast_neff_matrix"); plt.show()
valid = METRICS.loc[METRICS.valid_window]
display(pd.DataFrame([{"baseline": "Raw Leiden", "state median [Q1,Q3]": compact_iqr(valid["neff_raw"]), "Recon minus baseline": compact_iqr(valid["delta_neff_vs_raw_leiden"])}, {"baseline": "Raw Level2", "state median [Q1,Q3]": compact_iqr(valid["neff_level2"]), "Recon minus baseline": compact_iqr(valid["delta_neff_vs_raw_level2"])}]))
display(Markdown(f"Reconstructed Neff is **{compact_iqr(valid['neff_recon'])}**. The two delta maps separate reconstruction-associated differences from dependence on the chosen Raw baseline."))

### Evenness: balance conditional on observed richness

`evenness = Neff / Kobs` ranges from dominance toward balanced coexistence. It distinguishes windows with many labels but one dominant subtype from windows where those labels have comparable abundance.

In [ ]:
fig = plot_metric_comparison(RUN, "evenness", "Fibroblast: evenness evidence matrix")
save_figure(fig, "fibroblast_evenness_matrix"); plt.show()
valid = METRICS.loc[METRICS.valid_window]
display(pd.DataFrame([{"baseline": "Raw Leiden", "state median [Q1,Q3]": compact_iqr(valid["evenness_raw"]), "Recon minus baseline": compact_iqr(valid["delta_evenness_vs_raw_leiden"])}, {"baseline": "Raw Level2", "state median [Q1,Q3]": compact_iqr(valid["evenness_level2"]), "Recon minus baseline": compact_iqr(valid["delta_evenness_vs_raw_level2"])}]))
display(Markdown(f"Reconstructed evenness is **{compact_iqr(valid['evenness_recon'])}**; interpret it together with Kobs, because a pure one-cluster window also has evenness 1."))

### High-diversity Region

The Region is a parent-specific mask derived only from the data-driven breakpoint of reconstructed Neff. It is displayed beside the continuous Neff field and is not overlaid on anatomy.

In [ ]:
fig = plot_high_diversity(RUN, "Fibroblast: reconstructed diversity state and Region")
save_figure(fig, "fibroblast_high_diversity_region"); plt.show()
extent = extent_table(IMPACT, "in_state_region").loc[:, ["level1_region", "region_windows", "valid_windows", "region_area_um2", "area_fraction", "unit_fraction"]]
display(extent.loc[extent.level1_region.isin(["Overall", "Tumor", "Normal", "Interface"])])
display(Markdown(f"The reconstructed-Neff breakpoint is **{IMPACT.state_threshold.get('threshold')}** (status: **{IMPACT.state_threshold.get('status')}**). The anatomy rows describe context after Region definition; anatomy does not enter the threshold."))

## 6. Mono_Macro internal diversity and Regions

This parent is analysed after the fixed macro anatomy. Raw Level2 is mapped from the original Raw expression after Level1 parent selection; reconstructed-carrier Level2 labels are not used as this baseline.

### Cohort and window decision

The cell-equivalent side is fixed at 8 µm. The selected side comes only from parent occupancy; support is four units, and each of 200 paired draws samples four units.

In [ ]:
RUN = RUNS["Mono_Macro"]; IMPACT = RUN["impact"]
fig = plot_support_curve(IMPACT, "Mono_Macro: occupancy-only window decision")
save_figure(fig, "mono_macro_window_decision"); plt.show()
occupancy = IMPACT.unit_assignments.groupby("window_id").size()
sampled = RUN.get("sampled", RUN["used"])
decision = pd.DataFrame([{"available units": RUN["available"], "sampled units": sampled, "Raw-QC retained units": RUN["used"], "sampling mode": "full" if RUN["available"] == sampled else "deterministic 30k", "seed": CONFIG["partition_change"]["random_state"], "window side (cells per side)": IMPACT.scale_audit["main_window_cells_per_side"], "window side (µm)": IMPACT.scale_audit["main_window_side_um"], "occupancy median [Q1,Q3]": compact_iqr(occupancy), "minimum support": IMPACT.scale_audit["min_parent_units"], "rarefaction units": IMPACT.scale_audit["min_parent_units"], "paired draws": IMPACT.scale_audit["rarefaction_draws"]}])
display(decision)

### Internal-state baselines and matched-K assignments

The uniform Raw Level1 label is only the unexpanded baseline (`Kobs = Neff = evenness = 1`). The two informative Raw baselines are (i) expression-derived matched-K Leiden and (ii) reference-derived Raw Level2. Both are compared with the same reconstructed assignment on identical windows and identical rarefaction draws.

In [ ]:
METRICS = RUN["impact"].window_metrics.copy(); METRICS.attrs["side_um"] = IMPACT.scale_audit["main_window_side_um"]
fig = plot_reconstructed_clusters(RUN, "Mono_Macro: reconstructed internal state")
save_figure(fig, "mono_macro_reconstructed_clusters"); plt.show()
display(pd.DataFrame([{"baseline": "Raw Level1 (uniform parent)", "Kobs": 1.0, "Neff": 1.0, "evenness": 1.0}, {"baseline": "Raw Level2 mapping", "Kobs": RUN["raw_level2"].audit["n_mapped_level2"], "Neff": "window-specific", "evenness": "window-specific"}]))
display(pd.DataFrame([RUN["raw_level2"].audit]).loc[:, ["source", "method", "n_raw_units", "n_reference_cells", "n_reference_level2", "n_mapped_level2", "n_missing"]])

### Matched-K reconstruction-associated internal structure

This is a **parent-internal reassignment** analysis, not a Level1 identity change. Raw Leiden is tuned to approximately the same cluster count as the reconstructed/final assignment. Hungarian matching controls label permutation for unit-change metrics; the diversity matrices below use the original cluster compositions within each window.

In [ ]:
fig = plot_cluster_pair(RUN, "Mono_Macro: matched-K cluster assignment")
save_figure(fig, "mono_macro_matched_clusters"); plt.show()
row = RUN["comparison"].summary.iloc[0]
display(pd.DataFrame([{"paired units": row.n_units, "Raw K": row.n_raw_clusters, "Recon K": row.n_recon_clusters, "parent-internal reassignment": row.st_unit_change_fraction, "balanced change": row.balanced_cluster_change, "ARI": row["ARI"]}]).round(4))

### Kobs: local subtype richness

`Kobs` is the number of cluster labels observed in a rarefied window. Here it asks how many distinct internal states coexist locally, without accounting for whether one state dominates.

In [ ]:
fig = plot_metric_comparison(RUN, "k_obs", "Mono_Macro: Kobs evidence matrix")
save_figure(fig, "mono_macro_kobs_matrix"); plt.show()
valid = METRICS.loc[METRICS.valid_window]
display(pd.DataFrame([{"baseline": "Raw Leiden", "state median [Q1,Q3]": compact_iqr(valid["k_obs_raw"]), "Recon minus baseline": compact_iqr(valid["delta_k_obs_vs_raw_leiden"])}, {"baseline": "Raw Level2", "state median [Q1,Q3]": compact_iqr(valid["k_obs_level2"]), "Recon minus baseline": compact_iqr(valid["delta_k_obs_vs_raw_level2"])}]))
display(Markdown(f"Across valid windows, reconstructed Kobs is **{compact_iqr(valid['k_obs_recon'])}**. Its median difference is **{valid['delta_k_obs_vs_raw_leiden'].median():.3g}** versus Raw Leiden and **{valid['delta_k_obs_vs_raw_level2'].median():.3g}** versus Raw Level2."))

### Neff: abundance-aware effective subtype count

`Neff = exp(Shannon entropy)` converts subtype composition into an effective number of equally abundant clusters. It is the primary local-diversity state because it discounts rare labels and single-cluster dominance.

In [ ]:
fig = plot_metric_comparison(RUN, "neff", "Mono_Macro: Neff evidence matrix")
save_figure(fig, "mono_macro_neff_matrix"); plt.show()
valid = METRICS.loc[METRICS.valid_window]
display(pd.DataFrame([{"baseline": "Raw Leiden", "state median [Q1,Q3]": compact_iqr(valid["neff_raw"]), "Recon minus baseline": compact_iqr(valid["delta_neff_vs_raw_leiden"])}, {"baseline": "Raw Level2", "state median [Q1,Q3]": compact_iqr(valid["neff_level2"]), "Recon minus baseline": compact_iqr(valid["delta_neff_vs_raw_level2"])}]))
display(Markdown(f"Reconstructed Neff is **{compact_iqr(valid['neff_recon'])}**. The two delta maps separate reconstruction-associated differences from dependence on the chosen Raw baseline."))

### Evenness: balance conditional on observed richness

`evenness = Neff / Kobs` ranges from dominance toward balanced coexistence. It distinguishes windows with many labels but one dominant subtype from windows where those labels have comparable abundance.

In [ ]:
fig = plot_metric_comparison(RUN, "evenness", "Mono_Macro: evenness evidence matrix")
save_figure(fig, "mono_macro_evenness_matrix"); plt.show()
valid = METRICS.loc[METRICS.valid_window]
display(pd.DataFrame([{"baseline": "Raw Leiden", "state median [Q1,Q3]": compact_iqr(valid["evenness_raw"]), "Recon minus baseline": compact_iqr(valid["delta_evenness_vs_raw_leiden"])}, {"baseline": "Raw Level2", "state median [Q1,Q3]": compact_iqr(valid["evenness_level2"]), "Recon minus baseline": compact_iqr(valid["delta_evenness_vs_raw_level2"])}]))
display(Markdown(f"Reconstructed evenness is **{compact_iqr(valid['evenness_recon'])}**; interpret it together with Kobs, because a pure one-cluster window also has evenness 1."))

### High-diversity Region

The Region is a parent-specific mask derived only from the data-driven breakpoint of reconstructed Neff. It is displayed beside the continuous Neff field and is not overlaid on anatomy.

In [ ]:
fig = plot_high_diversity(RUN, "Mono_Macro: reconstructed diversity state and Region")
save_figure(fig, "mono_macro_high_diversity_region"); plt.show()
extent = extent_table(IMPACT, "in_state_region").loc[:, ["level1_region", "region_windows", "valid_windows", "region_area_um2", "area_fraction", "unit_fraction"]]
display(extent.loc[extent.level1_region.isin(["Overall", "Tumor", "Normal", "Interface"])])
display(Markdown(f"The reconstructed-Neff breakpoint is **{IMPACT.state_threshold.get('threshold')}** (status: **{IMPACT.state_threshold.get('status')}**). The anatomy rows describe context after Region definition; anatomy does not enter the threshold."))

## 7. T internal diversity and Regions

This parent is analysed after the fixed macro anatomy. Raw Level2 is mapped from the original Raw expression after Level1 parent selection; reconstructed-carrier Level2 labels are not used as this baseline.

### Cohort and window decision

The cell-equivalent side is fixed at 8 µm. The selected side comes only from parent occupancy; support is four units, and each of 200 paired draws samples four units.

In [ ]:
RUN = RUNS["T"]; IMPACT = RUN["impact"]
fig = plot_support_curve(IMPACT, "T: occupancy-only window decision")
save_figure(fig, "t_window_decision"); plt.show()
occupancy = IMPACT.unit_assignments.groupby("window_id").size()
sampled = RUN.get("sampled", RUN["used"])
decision = pd.DataFrame([{"available units": RUN["available"], "sampled units": sampled, "Raw-QC retained units": RUN["used"], "sampling mode": "full" if RUN["available"] == sampled else "deterministic 30k", "seed": CONFIG["partition_change"]["random_state"], "window side (cells per side)": IMPACT.scale_audit["main_window_cells_per_side"], "window side (µm)": IMPACT.scale_audit["main_window_side_um"], "occupancy median [Q1,Q3]": compact_iqr(occupancy), "minimum support": IMPACT.scale_audit["min_parent_units"], "rarefaction units": IMPACT.scale_audit["min_parent_units"], "paired draws": IMPACT.scale_audit["rarefaction_draws"]}])
display(decision)

### Internal-state baselines and matched-K assignments

The uniform Raw Level1 label is only the unexpanded baseline (`Kobs = Neff = evenness = 1`). The two informative Raw baselines are (i) expression-derived matched-K Leiden and (ii) reference-derived Raw Level2. Both are compared with the same reconstructed assignment on identical windows and identical rarefaction draws.

In [ ]:
METRICS = RUN["impact"].window_metrics.copy(); METRICS.attrs["side_um"] = IMPACT.scale_audit["main_window_side_um"]
fig = plot_reconstructed_clusters(RUN, "T: reconstructed internal state")
save_figure(fig, "t_reconstructed_clusters"); plt.show()
display(pd.DataFrame([{"baseline": "Raw Level1 (uniform parent)", "Kobs": 1.0, "Neff": 1.0, "evenness": 1.0}, {"baseline": "Raw Level2 mapping", "Kobs": RUN["raw_level2"].audit["n_mapped_level2"], "Neff": "window-specific", "evenness": "window-specific"}]))
display(pd.DataFrame([RUN["raw_level2"].audit]).loc[:, ["source", "method", "n_raw_units", "n_reference_cells", "n_reference_level2", "n_mapped_level2", "n_missing"]])

### Matched-K reconstruction-associated internal structure

This is a **parent-internal reassignment** analysis, not a Level1 identity change. Raw Leiden is tuned to approximately the same cluster count as the reconstructed/final assignment. Hungarian matching controls label permutation for unit-change metrics; the diversity matrices below use the original cluster compositions within each window.

In [ ]:
fig = plot_cluster_pair(RUN, "T: matched-K cluster assignment")
save_figure(fig, "t_matched_clusters"); plt.show()
row = RUN["comparison"].summary.iloc[0]
display(pd.DataFrame([{"paired units": row.n_units, "Raw K": row.n_raw_clusters, "Recon K": row.n_recon_clusters, "parent-internal reassignment": row.st_unit_change_fraction, "balanced change": row.balanced_cluster_change, "ARI": row["ARI"]}]).round(4))

### Kobs: local subtype richness

`Kobs` is the number of cluster labels observed in a rarefied window. Here it asks how many distinct internal states coexist locally, without accounting for whether one state dominates.

In [ ]:
fig = plot_metric_comparison(RUN, "k_obs", "T: Kobs evidence matrix")
save_figure(fig, "t_kobs_matrix"); plt.show()
valid = METRICS.loc[METRICS.valid_window]
display(pd.DataFrame([{"baseline": "Raw Leiden", "state median [Q1,Q3]": compact_iqr(valid["k_obs_raw"]), "Recon minus baseline": compact_iqr(valid["delta_k_obs_vs_raw_leiden"])}, {"baseline": "Raw Level2", "state median [Q1,Q3]": compact_iqr(valid["k_obs_level2"]), "Recon minus baseline": compact_iqr(valid["delta_k_obs_vs_raw_level2"])}]))
display(Markdown(f"Across valid windows, reconstructed Kobs is **{compact_iqr(valid['k_obs_recon'])}**. Its median difference is **{valid['delta_k_obs_vs_raw_leiden'].median():.3g}** versus Raw Leiden and **{valid['delta_k_obs_vs_raw_level2'].median():.3g}** versus Raw Level2."))

### Neff: abundance-aware effective subtype count

`Neff = exp(Shannon entropy)` converts subtype composition into an effective number of equally abundant clusters. It is the primary local-diversity state because it discounts rare labels and single-cluster dominance.

In [ ]:
fig = plot_metric_comparison(RUN, "neff", "T: Neff evidence matrix")
save_figure(fig, "t_neff_matrix"); plt.show()
valid = METRICS.loc[METRICS.valid_window]
display(pd.DataFrame([{"baseline": "Raw Leiden", "state median [Q1,Q3]": compact_iqr(valid["neff_raw"]), "Recon minus baseline": compact_iqr(valid["delta_neff_vs_raw_leiden"])}, {"baseline": "Raw Level2", "state median [Q1,Q3]": compact_iqr(valid["neff_level2"]), "Recon minus baseline": compact_iqr(valid["delta_neff_vs_raw_level2"])}]))
display(Markdown(f"Reconstructed Neff is **{compact_iqr(valid['neff_recon'])}**. The two delta maps separate reconstruction-associated differences from dependence on the chosen Raw baseline."))

### Evenness: balance conditional on observed richness

`evenness = Neff / Kobs` ranges from dominance toward balanced coexistence. It distinguishes windows with many labels but one dominant subtype from windows where those labels have comparable abundance.

In [ ]:
fig = plot_metric_comparison(RUN, "evenness", "T: evenness evidence matrix")
save_figure(fig, "t_evenness_matrix"); plt.show()
valid = METRICS.loc[METRICS.valid_window]
display(pd.DataFrame([{"baseline": "Raw Leiden", "state median [Q1,Q3]": compact_iqr(valid["evenness_raw"]), "Recon minus baseline": compact_iqr(valid["delta_evenness_vs_raw_leiden"])}, {"baseline": "Raw Level2", "state median [Q1,Q3]": compact_iqr(valid["evenness_level2"]), "Recon minus baseline": compact_iqr(valid["delta_evenness_vs_raw_level2"])}]))
display(Markdown(f"Reconstructed evenness is **{compact_iqr(valid['evenness_recon'])}**; interpret it together with Kobs, because a pure one-cluster window also has evenness 1."))

### High-diversity Region

The Region is a parent-specific mask derived only from the data-driven breakpoint of reconstructed Neff. It is displayed beside the continuous Neff field and is not overlaid on anatomy.

In [ ]:
fig = plot_high_diversity(RUN, "T: reconstructed diversity state and Region")
save_figure(fig, "t_high_diversity_region"); plt.show()
extent = extent_table(IMPACT, "in_state_region").loc[:, ["level1_region", "region_windows", "valid_windows", "region_area_um2", "area_fraction", "unit_fraction"]]
display(extent.loc[extent.level1_region.isin(["Overall", "Tumor", "Normal", "Interface"])])
display(Markdown(f"The reconstructed-Neff breakpoint is **{IMPACT.state_threshold.get('threshold')}** (status: **{IMPACT.state_threshold.get('status')}**). The anatomy rows describe context after Region definition; anatomy does not enter the threshold."))

## 8. Cross-parent summary and evidence boundary

The summary compares observations within each route and parent. Region masks use route- and parent-specific data-driven thresholds, so their fractions are not direct cross-platform biological comparisons.

In [ ]:
summary=[]
for parent, run in RUNS.items():
    row=run["comparison"].summary.iloc[0]; metrics=run["impact"].window_metrics.loc[lambda x: x.valid_window]
    region=extent_table(run["impact"], "in_state_region").query("level1_region == 'Overall'")
    summary.append({"parent": parent, "matched-K status": run["partition"].matched_cluster_status, "parent-internal reassignment": row.st_unit_change_fraction, "ARI": row["ARI"], "median ΔNeff vs Raw Leiden": metrics["delta_neff_vs_raw_leiden"].median(), "median ΔNeff vs Raw Level2": metrics["delta_neff_vs_raw_level2"].median(), "High-diversity unit fraction": region["unit_fraction"].iloc[0]})
summary=pd.DataFrame(summary); display(summary.round(4))
display(Markdown("The notebook separates global Level1-stratified change from parent-internal matched-K reassignment and diversity under two Raw baselines. High-diversity Regions are reconstructed-state descriptions, not anatomy, mechanism, pathology, or biological validation."))